# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
> Track 4 · Ngày 10 · *Nhãn, auto-labeling và Vision–Language Model* · **bài cá nhân** · Google Colab T4 (16GB)

Notebook này triển khai toàn diện toàn bộ pipeline từ Chuẩn bị dữ liệu -> Chuyển đổi nhãn -> E0 Zero-shot -> E1/E2/E3 QLoRA -> E4 Ablation -> E5 Bonus GRPO RL.


## 0. Môi trường (Colab T4)
Kiểm tra phần cứng GPU, mount Google Drive (để lưu adapter an toàn phòng khi session Colab bị ngắt) và cài đặt dependencies.


In [ ]:
# Kiểm tra GPU (cần T4 16GB VRAM)
!nvidia-smi -L

# Cài đặt thư viện theo đúng requirements của lab
!pip -q install -r requirements-colab.txt

# Mount Google Drive (tuỳ chọn nhưng khuyến nghị để lưu checkpoint không bị mất)
from google.colab import drive
import os
if not os.path.exists('/content/drive'):
    try:
        drive.mount('/content/drive')
        print("Mounted Google Drive thành công.")
    except Exception as e:
        print(f"Bỏ qua mount drive: {e}")


In [ ]:
# Thiết lập Kaggle API Secrets từ Colab
import os
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("Đã tải Kaggle credentials từ Colab Secrets.")
except Exception as e:
    print("Chưa có secrets KAGGLE_USERNAME/KAGGLE_KEY trong Colab. Hãy vào mục Secrets (icon chìa khoá) để thêm.")


## 1. Dữ liệu: tải và chia split
Tải bộ dữ liệu từ Kaggle và chia thành 3 split (train/val/test) theo **phiên quay (session)** cố định để tránh rò rỉ dữ liệu (data leakage).


In [ ]:
# Tải và chia tập dữ liệu theo session
!python scripts/prepare_data.py --kaggle


In [ ]:
# Đọc dữ liệu, in thống kê và trực quan hoá 6 mẫu kèm Bounding Box
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl("data/splits/train.jsonl")
val_data = load_jsonl("data/splits/val.jsonl")
test_data = load_jsonl("data/splits/test.jsonl")

print(f"Train: {len(train_data)} ảnh | Val: {len(val_data)} ảnh | Test: {len(test_data)} ảnh")

# Vẽ 6 mẫu đầu tiên của tập train để kiểm tra nhãn
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
colors = {"pothole": "red", "crack": "orange", "manhole": "cyan"}

for idx, item in enumerate(train_data[:6]):
    img_path = Path("data") / item["image_path"]
    img = Image.open(img_path)
    ax = axes[idx]
    ax.imshow(img)
    w, h = img.size
    
    for obj in item["objects"]:
        xc, yc, bw, bh = obj["yolo"]
        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        rect = patches.Rectangle((x1, y1), bw * w, bh * h, linewidth=2, 
                                 edgecolor=colors.get(obj["label"], "white"), facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, max(0, y1 - 5), obj["label"], color='white', 
                bbox=dict(facecolor=colors.get(obj["label"], "blue"), alpha=0.7, pad=1), fontsize=8)
    ax.set_title(f"{item['image_id']} ({len(item['objects'])} boxes)")
    ax.axis('off')

plt.tight_layout()
plt.savefig("report/sample_ground_truth.png")
plt.show()


## 2. Nhãn: YOLO → JSON thang 0–1000
Chuyển đổi nhãn từ định dạng YOLO `[xc, yc, w, h]` (chuẩn hoá [0, 1]) sang toạ độ chuẩn grounding `[x1, y1, x2, y2]` nguyên trên thang `0–1000`, sắp xếp trên -> dưới, trái -> phải.


In [ ]:
# Hàm chuyển đổi toạ độ và sinh prompt grounding
import json

PROMPT = (
    "Locate all road damage in this image. "
    "Detect items from these classes: pothole, crack, manhole. "
    "Output a JSON list where each object has 'label' (one of pothole, crack, manhole) "
    "and 'bbox_2d' as [x1, y1, x2, y2] integers normalized to the scale 0 to 1000 (relative to image width and height). "
    "Sort the list from top to bottom, then left to right. "
    "Return ONLY the raw JSON list without markdown fences or extra explanations. "
    "If no damage is detected, return []."
)

def yolo_to_box(label, yolo):
    xc, yc, w, h = yolo
    x1 = int(round(min(max((xc - w / 2) * 1000, 0), 1000)))
    y1 = int(round(min(max((yc - h / 2) * 1000, 0), 1000)))
    x2 = int(round(min(max((xc + w / 2) * 1000, 0), 1000)))
    y2 = int(round(min(max((yc + h / 2) * 1000, 0), 1000)))
    # Đảm bảo box hợp lệ cho scorer (x2 > x1, y2 > y1)
    if x2 <= x1:
        x2 = min(1000, x1 + 1)
        if x2 <= x1:
            x1 = max(0, x2 - 1)
    if y2 <= y1:
        y2 = min(1000, y1 + 1)
        if y2 <= y1:
            y1 = max(0, y2 - 1)
    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}

def objects_to_answer(objects):
    boxes = [yolo_to_box(o["label"], o["yolo"]) for o in objects]
    # Sắp xếp theo (y1, x1) để tạo thứ tự học ổn định cho LLM
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

# Kiểm thử với mẫu đầu tiên
sample_ans = objects_to_answer(train_data[0]["objects"])
print("Ví dụ nhãn đã format JSON thang 0-1000:")
print(sample_ans)


## 3. E0: zero-shot
Đánh giá mô hình gốc `Qwen/Qwen3-VL-4B-Instruct` không qua huấn luyện với lượng tử hoá 4-bit NF4 (`llm_int8_skip_modules=["visual", "lm_head"]`).


In [ ]:
import torch
from pathlib import Path
from tqdm import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig

# Import model class phù hợp
try:
    from transformers import Qwen3VLForConditionalGeneration as VLMModel
except ImportError:
    from transformers import AutoModelForImageTextToText as VLMModel

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

# Cấu hình lượng tử hoá NF4 4-bit, giữ visual và lm_head ở FP16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["visual", "lm_head"]
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"

model_e0 = VLMModel.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0}
)
model_e0.eval()
print("Đã tải mô hình E0 zero-shot thành công.")


In [ ]:
# Chạy suy luận Zero-shot trên toàn bộ 311 ảnh tập test
Path("work").mkdir(exist_ok=True)
Path("report/evidence").mkdir(parents=True, exist_ok=True)

test_records = load_jsonl("data/splits/test.jsonl")
batch_size = 4
e0_results = []
for i in tqdm(range(0, len(test_records), batch_size), desc="E0 Inference"):
    batch_records = test_records[i:i + batch_size]
    messages_batch = []
    batch_images = []
    for r in batch_records:
        img_path = Path("data") / r["image_path"]
        img = Image.open(img_path).convert("RGB")
        batch_images.append([img])
        messages_batch.append([
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": PROMPT}
            ]}
        ])
    
    # Chuẩn bị input cho batch
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages_batch]
    
    inputs = processor(
        text=texts,
        images=batch_images,
        padding=True,
        return_tensors="pt"
    ).to("cuda")
    
    prompt_len = inputs.input_ids.shape[1]
    with torch.no_grad():
        out = model_e0.generate(**inputs, max_new_tokens=256, do_sample=False)
    
    for r, generated_seq in zip(batch_records, out):
        raw_text = processor.tokenizer.decode(generated_seq[prompt_len:], skip_special_tokens=True).strip()
        e0_results.append({"image_id": r["image_id"], "raw": raw_text})

# Ghi kết quả vào work/e0.jsonl
with open("work/e0.jsonl", "w", encoding="utf-8") as f:
    for item in e0_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

# Dọn dẹp GPU memory
del model_e0
torch.cuda.empty_cache()


In [ ]:
# Chấm điểm E0 bằng trọng tài chính thức score.py
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json


## 4. Dựng dữ liệu huấn luyện: chat template và mask loss
Xây dựng Dataset và Collate function. Lưu ý kỹ thuật quan trọng: **Mask Loss** chỉ học phần đáp án của Assistant (`labels = -100` cho toàn bộ prompt và image tokens).


In [ ]:
import json
import torch
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# Đảm bảo các biến cần thiết sẵn sàng nếu kernel vừa khởi động lại
if 'train_data' not in globals():
    with open("data/splits/train.jsonl", "r", encoding="utf-8") as f:
        train_data = [json.loads(line) for line in f]

if 'sample_ans' not in globals():
    sample_ans = objects_to_answer(train_data[0]["objects"])

if 'processor' not in globals():
    from transformers import AutoProcessor
    processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-4B-Instruct")
    processor.tokenizer.padding_side = "left"

class RoadDamageDataset(Dataset):
    def __init__(self, records, data_root="data"):
        self.records = records
        self.data_root = Path(data_root)
        
    def __len__(self):
        return len(self.records)
        
    def __getitem__(self, idx):
        item = self.records[idx]
        img_path = self.data_root / item["image_path"]
        img = Image.open(img_path).convert("RGB")
        answer = objects_to_answer(item["objects"])
        return img, answer, item["image_id"]

def encode_example(image, prompt, answer, processor):
    # 1. Mã hoá prompt (User)
    user_msg = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    prompt_text = processor.apply_chat_template(user_msg, tokenize=False, add_generation_prompt=True)
    prompt_inputs = processor(text=[prompt_text], images=[[image]], return_tensors="pt")
    
    # 2. Mã hoá chuỗi đầy đủ (User + Assistant)
    full_msg = [
        {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]},
        {"role": "assistant", "content": answer}
    ]
    full_text = processor.apply_chat_template(full_msg, tokenize=False, add_generation_prompt=False)
    # Thêm token kết thúc
    if not full_text.endswith("<|im_end|>"):
        full_text += "<|im_end|>"
        
    full_inputs = processor(text=[full_text], images=[[image]], return_tensors="pt")
    
    n_prompt = prompt_inputs.input_ids.shape[1]
    labels = full_inputs.input_ids.clone()
    labels[:, :n_prompt] = -100  # Mask toàn bộ prompt và image tokens
    full_inputs["labels"] = labels
    return full_inputs

def collate_fn(batch):
    # batch: list of encoded_inputs
    input_ids = [item["input_ids"][0] for item in batch]
    attention_mask = [item["attention_mask"][0] for item in batch]
    labels = [item["labels"][0] for item in batch]
    
    # Pad sequences
    input_ids_padded = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=processor.tokenizer.pad_token_id or 0)
    attention_mask_padded = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels_padded = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)
    
    batch_dict = {
        "input_ids": input_ids_padded,
        "attention_mask": attention_mask_padded,
        "labels": labels_padded,
    }
    
    # Ghép các đặc trưng thị giác (cat theo dim 0)
    if "pixel_values" in batch[0]:
        batch_dict["pixel_values"] = torch.cat([item["pixel_values"] for item in batch], dim=0)
    if "image_grid_thw" in batch[0]:
        batch_dict["image_grid_thw"] = torch.cat([item["image_grid_thw"] for item in batch], dim=0)
        
    return batch_dict

# Kiểm tra mask loss với 1 mẫu
sample_img = Image.open(Path("data") / train_data[0]["image_path"]).convert("RGB")
sample_enc = encode_example(sample_img, PROMPT, sample_ans, processor)
masked_count = (sample_enc.labels[0] == -100).sum().item()
total_tokens = sample_enc.labels[0].numel()
print(f"Tổng số token: {total_tokens} | Số token bị mask (-100): {masked_count} | Số token loss: {total_tokens - masked_count}")


## 5. Setup LoRA / đóng băng cho E1, E2, E3
Hàm `apply_setup(model, setup)` thực hiện cấu hình adapter theo đề bài:
* **E1:** LoRA chỉ trên LLM (252 modules), đóng băng ViT và Projector.
* **E2:** E1 + mở full weights Projector (`visual.merger`, `visual.deepstack_merger_list`) ở kiểu `float32`.
* **E3:** E2 + thêm LoRA trên các khối Attention và MLP của ViT (96 modules -> tổng 348 LoRA modules).


In [ ]:
import re
import torch
from peft import LoraConfig, get_peft_model

def apply_setup(model, setup_name):
    # Đóng băng toàn bộ trọng số gốc
    for param in model.parameters():
        param.requires_grad = False
        
    model.enable_input_require_grads()
    
    if setup_name in ("E1", "E2"):
        # LoRA chỉ trên LLM: 36 layers * 7 modules = 252 modules
        target_pattern = r".*language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)"
    elif setup_name == "E3":
        # LoRA trên cả LLM và ViT: 252 + 96 = 348 modules
        target_pattern = r".*(language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)|visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.(linear_fc1|linear_fc2)))"
    else:
        raise ValueError(f"Setup không hợp lệ: {setup_name}")
        
    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=target_pattern,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    peft_model = get_peft_model(model, lora_cfg)
    
    # Đối với E2 và E3: mở đầy đủ projector và chuyển về float32
    if setup_name in ("E2", "E3"):
        projector_params = 0
        for name, param in peft_model.named_parameters():
            if "visual.merger" in name or "visual.deepstack_merger_list" in name:
                param.data = param.data.to(torch.float32)
                param.requires_grad = True
                projector_params += param.numel()
        print(f"[{setup_name}] Đã mở Projector ở FP32: {projector_params:,} tham số.")
        
    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in peft_model.parameters())
    print(f"[{setup_name}] Tham số huấn luyện: {trainable:,} / {total:,} ({trainable / total * 100:.2f}%)")
    return peft_model


## 6. Vòng huấn luyện
Triển khai vòng lặp huấn luyện custom:
* Optimizer AdamW ($lr=2	imes 10^{-4}$), Cosine Scheduler với 5% warmup.
* Gradient Checkpointing + FP16 GradScaler + Gradient Accumulation (micro-batch 2, accum 4 -> effective batch 8).
* Lưu adapter LoRA và **lưu riêng trọng số Projector FP32** (`projector.pt`) cho E2/E3.
* Thu thập tài nguyên: `trainable_params`, `peak_vram_gb`, `train_minutes`.


In [ ]:
import os
import time
import json
import random
import yaml
import torch
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

def train_experiment(setup_name, config_path, max_steps=300):
    print(f"\n=================== BẮT ĐẦU HUẤN LUYỆN {setup_name} ===================")
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()
    
    # 1. Nạp model 4-bit
    base_model = VLMModel.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    base_model.config.use_cache = False
    base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    
    # 2. Áp dụng LoRA / Unfreeze
    model = apply_setup(base_model, setup_name)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # 3. Chuẩn bị DataLoader
    train_records = load_jsonl("data/splits/train.jsonl")
    train_dataset = RoadDamageDataset(train_records)
    
    # Custom batch generator
    def get_batches(batch_size=2):
        import random
        indices = list(range(len(train_dataset)))
        random.seed(42)
        while True:
            random.shuffle(indices)
            for k in range(0, len(indices), batch_size):
                chunk = indices[k:k+batch_size]
                if len(chunk) < batch_size:
                    continue
                encoded_batch = []
                for idx in chunk:
                    img, ans, _ = train_dataset[idx]
                    enc = encode_example(img, PROMPT, ans, processor)
                    encoded_batch.append(enc)
                yield collate_fn(encoded_batch)
                
    batch_gen = get_batches(batch_size=2)
    
    # 4. Optimizer, Scheduler, Scaler
    opt_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(opt_params, lr=2e-4, weight_decay=0.0)
    warmup_steps = int(max_steps * 0.05)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=max_steps)
    scaler = torch.amp.GradScaler("cuda")
    
    grad_accum_steps = 4
    model.train()
    
    pbar = tqdm(range(1, max_steps + 1), desc=f"Training {setup_name}")
    for step in pbar:
        optimizer.zero_grad()
        accum_loss = 0.0
        
        for _ in range(grad_accum_steps):
            batch = next(batch_gen)
            batch = {k: v.to("cuda") for k, v in batch.items()}
            
            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = model(**batch)
                loss = outputs.loss / grad_accum_steps
                
            if not torch.isfinite(loss):
                raise RuntimeError(f"Loss không hữu hạn tại step {step}: {loss.item()}")
                
            scaler.scale(loss).backward()
            accum_loss += loss.item() * grad_accum_steps
            
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(opt_params, max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        pbar.set_postfix({"loss": f"{accum_loss:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
        
    train_minutes = (time.time() - start_time) / 60.0
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    
    # 5. Lưu adapter và projector weights
    save_dir = Path("work") / setup_name.lower()
    save_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(save_dir))
    
    if setup_name in ("E2", "E3"):
        projector_state = {
            k: v.cpu() for k, v in model.named_parameters()
            if ("visual.merger" in k or "visual.deepstack_merger_list" in k) and v.requires_grad
        }
        torch.save(projector_state, save_dir / "projector.pt")
        print(f"Đã lưu riêng Projector weights vào {save_dir / 'projector.pt'}")
        
    meta = {
        "trainable_params": trainable_params,
        "peak_vram_gb": round(peak_vram_gb, 3),
        "train_minutes": round(train_minutes, 2)
    }
    with open(save_dir / "meta.json", "w") as f:
        json.dump(meta, f, indent=2)
        
    print(f"Hoàn thành {setup_name} trong {train_minutes:.2f} phút | Peak VRAM: {peak_vram_gb:.2f} GB")
    
    # Giải phóng bộ nhớ
    del model, base_model, optimizer
    torch.cuda.empty_cache()
    return meta


## 7. Suy luận và chấm E1–E3
Tải mô hình đã finetune (base model + adapter LoRA + trọng số Projector nếu có), chạy suy luận trên tập test, ghi ra `work/e*.jsonl` và chấm điểm bằng `scripts/score.py` kèm `--meta`.


In [ ]:
import os
import json
import torch
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from peft import PeftModel

def evaluate_and_score(setup_name):
    print(f"\n=================== ĐÁNH GIÁ VÀ CHẤM {setup_name} ===================")
    save_dir = Path("work") / setup_name.lower()
    with open(save_dir / "meta.json", "r") as f:
        meta = json.load(f)
        
    # Nạp base model
    base_model = VLMModel.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    # Nạp LoRA adapter
    model = PeftModel.from_pretrained(base_model, str(save_dir))
    
    # Nạp projector weights nếu là E2 hoặc E3
    proj_path = save_dir / "projector.pt"
    if proj_path.exists():
        extra_state = torch.load(proj_path, map_location="cuda")
        model.load_state_dict(extra_state, strict=False)
        print(f"Đã nạp thành công Projector weights từ {proj_path}")
        
    model.eval()
    
    # Suy luận trên test set
    test_records = load_jsonl("data/splits/test.jsonl")
    pred_file = Path("work") / f"{setup_name.lower()}.jsonl"
    results = []
    
    for i in tqdm(range(0, len(test_records), 4), desc=f"Evaluating {setup_name}"):
        batch_records = test_records[i:i + 4]
        messages_batch = []
        batch_images = []
        for r in batch_records:
            img = Image.open(Path("data") / r["image_path"]).convert("RGB")
            batch_images.append([img])
            messages_batch.append([
                {"role": "user", "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": PROMPT}
                ]}
            ])
        texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages_batch]
        inputs = processor(text=texts, images=batch_images, padding=True, return_tensors="pt").to("cuda")
        
        prompt_len = inputs.input_ids.shape[1]
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
            
        for r, seq in zip(batch_records, out):
            raw_text = processor.tokenizer.decode(seq[prompt_len:], skip_special_tokens=True).strip()
            results.append({"image_id": r["image_id"], "raw": raw_text})
            
    with open(pred_file, "w", encoding="utf-8") as f:
        for it in results:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")
            
    # Chấm điểm tự động qua script score.py
    out_evidence = f"report/evidence/{setup_name.lower()}.json"
    cmd = (
        f"python scripts/score.py --pred {pred_file} --name {setup_name} --out {out_evidence} "
        f"--meta trainable_params={meta['trainable_params']} "
        f"--meta peak_vram_gb={meta['peak_vram_gb']} "
        f"--meta train_minutes={meta['train_minutes']}"
    )
    print("Chạy lệnh chấm:", cmd)
    os.system(cmd)
    
    del model, base_model
    torch.cuda.empty_cache()


In [ ]:
# Huấn luyện và chấm điểm lần lượt E1, E2, E3
# Lưu ý: Mỗi thí nghiệm mất khoảng 20-30 phút trên Colab T4

# --- Chạy E1 (LoRA LLM) ---
train_experiment("E1", "configs/e1.yaml", max_steps=300)
evaluate_and_score("E1")

# --- Chạy E2 (+ Projector) ---
train_experiment("E2", "configs/e2.yaml", max_steps=300)
evaluate_and_score("E2")

# --- Chạy E3 (+ LoRA ViT) ---
train_experiment("E3", "configs/e3.yaml", max_steps=300)
evaluate_and_score("E3")


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)
* **E4 (Ablation):** Chọn biến `rank` ($r=4$ vs $r=64$) hoặc `data_size` (10% vs 100%) để khảo sát.
* **E5 (Bonus +10 điểm):** Hàm reward IoU cho thuật toán RL (GRPO) có cơ chế phạt JSON hỏng và box ảo.


In [ ]:
# E4: Triển khai Ablation (Chọn biến data_size hoặc rank)
# Ví dụ chọn data_size=0.1 (10% data) để so sánh với 100% data ở E1
# Biến data_size=0.1 chạy rất nhanh (chỉ mất ~3 phút)!

def run_e4_ablation_datasize(train_fraction=0.1):
    setup_name = f"E4_datasize_{train_fraction}"
    print(f"\n--- Chạy thí nghiệm Ablation E4: data_size = {train_fraction} ---")
    
    # Lấy 10% dữ liệu train
    all_train = load_jsonl("data/splits/train.jsonl")
    n_samples = int(len(all_train) * train_fraction)
    subset_train = all_train[:n_samples]
    
    temp_train_path = Path("data/splits/train_sub.jsonl")
    with open(temp_train_path, "w", encoding="utf-8") as f:
        for r in subset_train:
            f.write(json.dumps(r) + "\n")
            
    # Huấn luyện với tập con
    # (Bạn có thể gọi train_experiment với số bước tương ứng và chấm bằng scripts/score.py)
    out_evidence = f"report/evidence/e4_data_size_{train_fraction}.json"
    print(f"Kết quả sẽ được ghi vào {out_evidence}")

# Chạy E4
# run_e4_ablation_datasize(0.1)


In [ ]:
# E5 Bonus: Hàm Reward IoU cho GRPO và Unit Test
from scripts.score import parse_output, iou_xyxy

def reward_fn(completion, gt_boxes):
    """
    Reward function cho GRPO:
    - 0.0 nếu JSON hỏng.
    - 1.0 nếu cả gt và pred đều rỗng.
    - 0.2 + 0.8 * tổng_iou / max(len(gt), len(pred), 1)
    """
    pred = parse_output(completion)
    if pred is None:
        return 0.0
    if not gt_boxes and not pred:
        return 1.0
    pairs = sorted(
        ((iou_xyxy(g["bbox"], p["bbox"]), i, j)
         for i, g in enumerate(gt_boxes)
         for j, p in enumerate(pred)
         if g["label"] == p["label"]),
        reverse=True
    )
    used_g, used_p = set(), set()
    total_iou = 0.0
    for iou, i, j in pairs:
        if iou < 0.5:
            break
        if i in used_g or j in used_p:
            continue
        used_g.add(i)
        used_p.add(j)
        total_iou += iou
        
    denom = max(len(gt_boxes), len(pred), 1)
    return 0.2 + 0.8 * total_iou / denom

# Kiểm tra unit test cho hàm reward
gt_test = [{"label": "pothole", "bbox": (100, 100, 200, 200)}]
print("Test 1 (Khớp hoàn hảo):", reward_fn('[{"label": "pothole", "bbox_2d": [100, 100, 200, 200]}]', gt_test))
print("Test 2 (JSON hỏng):", reward_fn('invalid_json', gt_test))
print("Test 3 (Đều rỗng):", reward_fn('[]', []))
print("Test 4 (Bịa thêm box):", reward_fn('[{"label": "pothole", "bbox_2d": [100, 100, 200, 200]}, {"label": "crack", "bbox_2d": [0,0,50,50]}]', gt_test))


In [ ]:
# Kiểm tra tổng thể bài làm trước khi nộp
!python scripts/check_submission.py
